# Dunnhumby K=1 가치기저 M2·M5의 CLV 배정 순열 대조군
실제 CLV 배정과 degree 10분위 안에서 섞은 배정을 같은 실행에서 학습합니다. M2 가치블록만 순열하고 M4 가중치는 모든 arm에서 실제 q_C를 씁니다. seed 42, DAY 1~683 학습 → 684~690 개발평가, 100 epoch, 네 번 학습합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = 'd1d098f4573c14f13d83dbe0c3a5ef970bc0e67a'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA


In [ ]:
import json
import torch
import lightgcn_clv_m5_value_basis_k1_assignment_controls as controls

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
assert controls.CODE_VERSION == 'm5-value-basis-k1-clv-assignment-controls-v1'
cfg = controls.configure_assignment_controls()
summary = controls.preflight_summary(cfg)
assert summary['loss']['negative_count'] == 1
assert summary['m2']['rho'] == 0.25
assert summary['control']['permuted_component'] == 'M2 value block only'
assert summary['reused_models'] == []
print(json.dumps(summary, ensure_ascii=False, indent=2))


In [ ]:
result_df = controls.run_assignment_controls(cfg)


In [ ]:
from IPython.display import display

def show(frame):
    view = frame.copy()
    view.attrs = {}
    display(view)

core = ['model_id', 'm2_assignment', 'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20', 'recall@50', 'ndcg@50',
        'price_purchase_amount_weighted_hit@10', 'vndcg@10', 'user_value_tendency_recommended_price_alignment']
print('1) 실제 배정과 순열 배정의 절대지표')
show(result_df[core])
print('2) 순열 불변식')
print(json.dumps(result_df.attrs['control_diagnostics'], ensure_ascii=False, indent=2))
print('3) Top-10 변경 비율 (실제 vs 순열)')
show(result_df.attrs['top10_overlap'])
print('4) 경제점수 영향력')
show(result_df.attrs['score_diagnostics'])
print('5) 판독')
print(json.dumps(result_df.attrs['decision'], ensure_ascii=False, indent=2))
print('저장 파일:', json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))
